# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    REPO_DIR = "flyrank-ml-internship-starter"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if not IN_COLAB and os.getcwd().replace("\\", "/").endswith("work/notebooks"):
    os.chdir("../..")

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

pd.set_option("display.width", 120)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found -- are you at the repo root?"
print("Loaded:", df.shape)

# Working from the RAW csv (not the pipeline's fillna'd feature vector) so blanks show up as
# real NaN here, not an artificial zero -- a blind fillna(0) would inject a fake mass at 0 for
# columns like word_count where "missing" and "zero" mean different things.

fields = ["impressions_90d", "sessions_90d", "word_count", "days_since_last_update", "content_age_days"]
for col in fields:
    valid = df[col].dropna()
    n_missing = int(df[col].isna().sum())
    print(f"\n--- {col} (n={len(valid)}, missing={n_missing}) ---")
    print(valid.describe(percentiles=[.5, .75, .9, .95, .99]).to_string())
    if valid.median() > 0:
        print(f"mean/median ratio: {valid.mean() / valid.median():.2f}  (>1.5 signals a heavy right tail)")

# Heavy-tail check on the two traffic counts the pipeline log1p's -- compare raw vs. logged
for col in ["impressions_90d", "sessions_90d"]:
    raw = df[col].dropna()
    logged = np.log1p(raw)
    print(f"\n--- {col}: raw vs. log1p ---")
    print(f"raw:   mean={raw.mean():.1f}  median={raw.median():.1f}  p99={raw.quantile(.99):.1f}  max={raw.max():.1f}")
    print(f"log1p: mean={logged.mean():.2f}  median={logged.median():.2f}  p99={logged.quantile(.99):.2f}")
    print("mean/median much closer after log1p -- confirms why the pipeline models the logged column, not the raw count")

# avg_position: 0 means "no position data", not rank zero -- split before describing
no_position = df["avg_position"] == 0
print(f"\n--- avg_position ({no_position.sum()} rows / {no_position.mean() * 100:.1f}% show 0 = 'no data', excluded below) ---")
print(df.loc[~no_position, "avg_position"].describe(percentiles=[.5, .75, .9, .95, .99]).to_string())

# ctr is stored as a x100 percentage already (0.76 means 0.76%, not 76%)
print("\n--- ctr (already a x100 percentage: 0.76 = 0.76%, not 76%) ---")
print(df["ctr"].describe(percentiles=[.5, .75, .9, .95, .99]).to_string())

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
SAMPLE_FLOOR = 50


def trend_direction(values):
    diffs = [b - a for a, b in zip(values, values[1:])]
    if all(d > 0 for d in diffs):
        return "increasing"
    if all(d < 0 for d in diffs):
        return "decreasing"
    return "mixed"


def verdict(values, expect, n_values):
    if any(n < SAMPLE_FLOOR for n in n_values):
        return "INSUFFICIENT DATA (a bucket is under the 50-row floor)"
    direction = trend_direction(values)
    if direction == "mixed":
        return "MIXED"
    spread = abs(values[-1] - values[0]) / (abs(values[0]) if values[0] else 1)
    if direction == expect and spread > 0.05:
        return "CONFIRMED"
    if direction == expect:
        return "FALSE (direction matches but the gap is too small to call a real effect)"
    return "OPPOSITE"


# --- Signal 1: "Longer articles earn more organic search impressions" (word_count vs impressions_90d)
wc_order = ["<1000", "1000-2000", "2000-3500", "3500+"]
g1 = (
    df.dropna(subset=["word_count_tier"])
    .groupby("word_count_tier")["impressions_90d"]
    .agg(n="count", median_impressions="median")
    .reindex(wc_order)
)
print("--- Signal 1: word_count_tier vs. median impressions_90d ---")
print(g1.to_string())
v1 = verdict(g1["median_impressions"].tolist(), "increasing", g1["n"].tolist())
print(f"VERDICT: {v1}")

# --- Signal 2: "Better average position drives higher CTR" (position_tier vs. weighted ctr)
# weighted, not mean-of-per-row-rates: total_clicks / total_impressions per bucket
#
# Data-quality catch: ALL 1,205 avg_position==0 ("no position data") rows are mislabeled into
# position_tier=="top_3" upstream -- the tier build did not special-case the zero-means-no-data
# rule the way the raw avg_position column does. Filtering to avg_position > 0 below removes
# them; skipping that filter would falsely count 1,205 unranked pages as best-ranked.
mislabeled = ((df["position_tier"] == "top_3") & (df["avg_position"] == 0)).sum()
top3_total = (df["position_tier"] == "top_3").sum()
print(f"position_tier=='top_3': {top3_total} rows, {mislabeled} of them are actually avg_position==0 (no data)")

pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
g2 = (
    df[df["avg_position"] > 0]
    .groupby("position_tier")
    .agg(n=("ctr", "count"), total_impressions=("impressions_90d", "sum"), total_clicks=("clicks_90d", "sum"))
    .reindex(pos_order)
)
g2["weighted_ctr"] = g2["total_clicks"] / g2["total_impressions"] * 100
print("\n--- Signal 2: position_tier (best -> worst) vs. weighted CTR ---")
print(g2[["n", "weighted_ctr"]].to_string())
v2 = verdict(g2["weighted_ctr"].tolist(), "decreasing", g2["n"].tolist())
print(f"VERDICT: {v2}")

# --- Signal 3: "Content untouched for longer gets less traffic" (freshness_tier vs impressions_90d)
fresh_order = ["0-30", "31-90", "91-180", "181+"]
g3 = (
    df.groupby("freshness_tier")["impressions_90d"]
    .agg(n="count", median_impressions="median")
    .reindex(fresh_order)
)
print("\n--- Signal 3: freshness_tier vs. median impressions_90d ---")
print(g3.to_string())
v3 = verdict(g3["median_impressions"].tolist(), "decreasing", g3["n"].tolist())
print(f"VERDICT: {v3}")

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
# FlyRank's real flag being tested: `page_one_decay_risk` from scripts/02_baseline_score.py --
#   if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180
# Assumption behind the rule: among page-1 content, OLDER pages are more likely to be declining
# than younger page-1 content -- that's the "decay risk" the flag is betting on.
#
# Note this rule already guards avg_position > 0 -- it dodges the position_tier "no data
# mislabeled as top_3" trap found in section 2. Mirroring that guard here.

page_one = df[(df["avg_position"] > 0) & (df["avg_position"] <= 10)].copy()
page_one["age_group"] = np.where(page_one["content_age_days"] >= 180, "old (>=180d, flagged)", "young (<180d, not flagged)")

flag_test = page_one.groupby("age_group")["trend_direction"].apply(
    lambda s: (s.str.lower() == "down").mean()
).rename("declining_rate")
flag_n = page_one.groupby("age_group").size().rename("n")
flag_table = pd.concat([flag_n, flag_test], axis=1)
print("--- page_one_decay_risk: page-1 content (avg_position 0-10), old vs. young ---")
print(flag_table.to_string())

overall_base_rate = (df["trend_direction"].str.lower() == "down").mean()
print(f"\noverall base rate (all rows, not just page-1): {overall_base_rate * 100:.1f}%")

old_rate = flag_table.loc["old (>=180d, flagged)", "declining_rate"]
young_rate = flag_table.loc["young (<180d, not flagged)", "declining_rate"]
old_n = flag_table.loc["old (>=180d, flagged)", "n"]
young_n = flag_table.loc["young (<180d, not flagged)", "n"]

if old_n < SAMPLE_FLOOR or young_n < SAMPLE_FLOOR:
    flag_verdict = "INSUFFICIENT DATA (a group is under the 50-row floor)"
elif old_rate - young_rate > 0.05:
    flag_verdict = "CONFIRMED"
elif young_rate - old_rate > 0.05:
    flag_verdict = "OPPOSITE"
else:
    flag_verdict = "FALSE (gap too small to call a real effect)"

print(f"\nold declining rate {old_rate * 100:.1f}% (n={old_n}) vs. young declining rate {young_rate * 100:.1f}% (n={young_n})")
print(f"gap (old - young): {(old_rate - young_rate) * 100:+.1f} points")
print(f"VERDICT: {flag_verdict}")

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
# For a content team:
#
# Word count and position remain trustworthy signals -- content depth tracks organic visibility
# (Signal 1, CONFIRMED) and position gains compound into CTR (Signal 2, CONFIRMED) -- but don't
# trust a "top_3" tag by itself; check avg_position > 0 first, since 1,205 unranked pages are
# mislabeled into that bucket by a tier-build bug.
#
# Staleness alone is not a reliable urgency signal (Signal 3, MIXED) -- don't triage purely by
# days_since_last_update.
#
# Most importantly: the page_one_decay_risk flag has the direction backwards in this data. Older
# page-1 content (>=180 days) is declining LESS often (51.8%) than younger page-1 content
# (61.7%) -- a 9.9-point gap the wrong way. That flag is currently pointing reviewers at the
# wrong half of the page-1 queue; worth raising with whoever owns that rule before trusting its
# "review these first" ranking.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.